# Abalone: Final Three Tree-Based Models (PCA)


## 1. Imports and configuration

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
)
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATA_PATH = Path("data/abalone_cleaned.csv")
TARGET = "Rings"
NUMERIC_FEATURES = [
    "Length", "Diameter", "Height", "Whole weight",
    "Shucked weight", "Viscera weight", "Shell weight",
]


## 2. Read the cleaned dataset

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
            f"Could not find {DATA_PATH.resolve()}. Put abalone_cleaned.csv beside this notebook "
            "or update DATA_PATH to the correct location."
        )

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
print("Missing values:", int(df.isna().sum().sum()))
print(df.head())

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print("Train:", X_train.shape[0], "Test:", X_test.shape[0])


## 3. PCA preprocessing (use only PC1 and PC2)


In [ ]:
preprocessor = ColumnTransformer([
    ("numeric", "passthrough", NUMERIC_FEATURES),
    ("sex", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), ["Sex"]),
])

def make_pca_model(estimator):
    return Pipeline([
        ("preprocessing", clone(preprocessor)),
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=2)),
        ("model", estimator),
    ])


## 4. Three tree-based models

In [ ]:
models = {
    "Random Forest": RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1),
    "Extra Trees": ExtraTreesRegressor(random_state=RANDOM_STATE, n_jobs=1),
    "XGBoost": XGBRegressor(random_state=RANDOM_STATE, n_jobs=1, objective="reg:squarederror"),
}

param_distributions = {
    "Random Forest": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [None, 10, 20],
        "model__min_samples_leaf": [1, 2, 4],
        "model__max_features": [0.7, 1.0],
    },
    "Extra Trees": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [None, 10, 20],
        "model__min_samples_leaf": [1, 2, 4],
        "model__max_features": [0.7, 1.0],
    },
    "XGBoost": {
        "model__learning_rate": [0.03, 0.1],
        "model__n_estimators": [100, 200],
        "model__max_depth": [3, 5],
        "model__subsample": [0.8, 1.0],
        "model__colsample_bytree": [0.8, 1.0],
    },
}


## 5. Five-fold CV and hyperparameter tuning


In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

best_models = {}
cv_rows = []

for name, estimator in models.items():
    search = RandomizedSearchCV(
        estimator=make_pca_model(estimator),
        param_distributions=param_distributions[name],
        n_iter=6,
        cv=cv,
        scoring="neg_mean_absolute_error",
        n_jobs=2,
        random_state=RANDOM_STATE,
        refit=True,
    )
    search.fit(X_train, y_train)

    best_models[name] = search.best_estimator_
    cv_rows.append({
        "Model": name,
        "CV_MAE": -search.best_score_,
        "Best_Parameters": str(search.best_params_),
    })
    print(name, "CV MAE:", round(-search.best_score_, 4))

cv_results = pd.DataFrame(cv_rows).sort_values("CV_MAE")
print("\nCross-validation results:")
print(cv_results.to_string(index=False))


### 5.1 Linear Regression baseline (same PCA and data split)

This adds a linear baseline without changing the three existing tree models. It uses the same two principal components, five-fold CV, and held-out test set.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

# Use the same preprocessing + PC1/PC2 as the three tree-based models.
linear_reg = make_pca_model(LinearRegression())

# Same five-fold CV; linear regression has no hyperparameters to tune here.
linear_cv_mae = -cross_val_score(
    linear_reg,
    X_train,
    y_train,
    cv=cv,
    scoring="neg_mean_absolute_error",
    n_jobs=2,
)

# Fit on the full training set, and include it in the existing test evaluation.
linear_reg.fit(X_train, y_train)
best_models["Linear Regression"] = linear_reg

cv_rows.append({
    "Model": "Linear Regression",
    "CV_MAE": linear_cv_mae.mean(),
    "Best_Parameters": "N/A (no tuning)",
})
cv_results = pd.DataFrame(cv_rows).sort_values("CV_MAE")

print("Linear Regression CV MAE:", round(linear_cv_mae.mean(), 4))
print("\nUpdated cross-validation results:")
print(cv_results.to_string(index=False))


## 6. Final test evaluation


In [ ]:
test_rows = []
predictions = pd.DataFrame({
    "original_row_index": X_test.index,
    "actual_rings": y_test.to_numpy(),
})

for name, model in best_models.items():
    y_pred = model.predict(X_test)
    predictions[name + "_predicted_rings"] = y_pred
    test_rows.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R2": r2_score(y_test, y_pred),
    })

test_results = pd.DataFrame(test_rows).sort_values("MAE")
print("Test results (lower MAE/RMSE is better; higher R2 is better):")
print(test_results.round(4).to_string(index=False))


## 7. Export results to share with teammates

In [ ]:
results_dir = Path("results")
results_dir.mkdir(parents=True, exist_ok=True)

cv_results.to_csv(results_dir / "pca_three_trees_cv.csv", index=False)
test_results.to_csv(results_dir / "pca_three_trees_test_metrics.csv", index=False)
predictions.to_csv(results_dir / "pca_three_trees_predictions.csv", index=False)

print("Saved:")
print(" -", results_dir / "pca_three_trees_cv.csv")
print(" -", results_dir / "pca_three_trees_test_metrics.csv")
print(" -", results_dir / "pca_three_trees_predictions.csv")
